# 🎤 Real-Time / Test Data Trimmer — Multi-Mode Audio Processor

A **3-in-1 trimmer** for extracting and classifying test data. Switch between modes:

| Mode | What It Does |
|------|--------------|
| `batch` | Process a folder of audio files |
| `single` | Analyze one file in detail |
| `microphone` | Capture live audio from your PC mic |

## How To Use
1. Set `MODE` in Cell 3
2. Set the input path for your chosen mode
3. Run all cells — only the active mode executes

See `manual/README.md` for full documentation including microphone setup.

In [ ]:
# ============================================================
# CELL 1: Install Dependencies
# ============================================================
%pip install -q librosa soundfile pandas numpy tqdm matplotlib sounddevice

In [ ]:
# ============================================================
# CELL 2: Imports
# ============================================================
import librosa
import librosa.display
import numpy as np
import pandas as pd
import soundfile as sf
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
import json
import shutil
import random
import re
import time
from datetime import datetime
import threading
import IPython.display as ipd

try:
    import sounddevice as sd
    SD_AVAILABLE = True
except ImportError:
    SD_AVAILABLE = False
    print('⚠️ sounddevice not available. Microphone mode disabled.')
    print('   Install with: pip install sounddevice')

warnings.filterwarnings('ignore')
np.random.seed(42)
random.seed(42)
print('✅ All imports loaded successfully.')

In [ ]:
# ============================================================
# CELL 3: CONFIGURATION — SET YOUR MODE AND PATHS HERE
# ============================================================

# ==========================================================
# MODE SELECTION — Change this to switch between modes
# ==========================================================
MODE = "batch"           # Options: "batch", "single", "microphone"

# ==========================================================
# PATH AUTO-DETECTION
# ==========================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'Data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

# !! TO HARDCODE: Uncomment below !!
# PROJECT_ROOT = Path(r'd:\Desktop\Data-Cleaner')

DATA_DIR = PROJECT_ROOT / 'Data'
OUTPUT_DIR = DATA_DIR / 'TRIMMED_REALTIME'

# ==========================================================
# BATCH MODE SETTINGS
# ==========================================================
# Folder containing .wav files to process:
BATCH_INPUT_DIR = DATA_DIR / 'gun'     # Change to your test data folder
BATCH_HOP_MS = 125                      # Sliding window hop for non-onset clips

# ==========================================================
# SINGLE FILE MODE SETTINGS
# ==========================================================
# Path to one .wav file to analyze in detail:
SINGLE_INPUT_FILE = None  # Set to a Path, e.g.:
# SINGLE_INPUT_FILE = DATA_DIR / 'gun' / 'BoltAction22_Samsung' / 'SA_004A_S01.wav'

# ==========================================================
# MICROPHONE MODE SETTINGS
# ==========================================================
MIC_DEVICE_INDEX = None          # None = default mic. Run sd.query_devices() to find yours.
MIC_BUFFER_SECONDS = 2.0         # Rolling buffer length
MIC_TRIGGER_RMS = 0.05           # RMS threshold to trigger capture
MIC_CAPTURE_DURATION_S = 30.0    # How long to listen (seconds). 0 = unlimited (Ctrl+C to stop).
MIC_COOLDOWN_S = 0.5             # Minimum seconds between captures

# ==========================================================
# AUDIO PARAMETERS (shared across all modes)
# ==========================================================
SAMPLE_RATE = 22050
TARGET_MS = 250
TARGET_SAMPLES = int(SAMPLE_RATE * TARGET_MS / 1000)  # = 5512
PRE_EVENT_MS = 50
PRE_EVENT_SAMPLES = int(SAMPLE_RATE * PRE_EVENT_MS / 1000)

# Classification thresholds (heuristic)
GUNSHOT_CREST_MIN = 4.0
GUNSHOT_PEAK_MIN = 0.02
GUNSHOT_CENTROID_MIN = 1500.0

OVERWRITE = True

# --- Print config ---
print(f'\n{"=" * 50}')
print(f'MODE: {MODE.upper()}')
print(f'{"=" * 50}')
print(f'Project Root  : {PROJECT_ROOT}')
print(f'Output        : {OUTPUT_DIR}')
print(f'Clip Duration : {TARGET_MS}ms ({TARGET_SAMPLES} samples)')
if MODE == 'batch':
    print(f'Batch Input   : {BATCH_INPUT_DIR}')
elif MODE == 'single':
    print(f'Single File   : {SINGLE_INPUT_FILE}')
elif MODE == 'microphone':
    print(f'Mic Device    : {MIC_DEVICE_INDEX or "default"}')
    print(f'Trigger RMS   : {MIC_TRIGGER_RMS}')
    print(f'Duration      : {MIC_CAPTURE_DURATION_S}s')
print(f'{"=" * 50}')

In [ ]:
# ============================================================
# CELL 4: Helper Functions (shared across all modes)
# ============================================================
JUNK_TOKENS = ('__MACOSX',)
SAFE_NAME_RE = re.compile(r'[^A-Za-z0-9._-]+')


def is_junk(path):
    as_str = str(path)
    return any(t in as_str for t in JUNK_TOKENS) or path.name.startswith('._')


def sanitize_name(raw, max_len=120):
    cleaned = SAFE_NAME_RE.sub('_', raw.strip()).strip('._')
    return (cleaned or 'clip')[:max_len]


def collect_wavs(dirs):
    if isinstance(dirs, (str, Path)):
        dirs = [Path(dirs)]
    files = []
    for d in dirs:
        d = Path(d)
        if d.exists():
            files.extend([f for f in d.rglob('*.wav') if not is_junk(f)])
    return sorted(files)


def load_audio(path):
    y, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)
    return y.astype(np.float32)


def force_exact_length(clip):
    if len(clip) == TARGET_SAMPLES:
        return clip
    if len(clip) > TARGET_SAMPLES:
        return clip[:TARGET_SAMPLES]
    return np.pad(clip, (0, TARGET_SAMPLES - len(clip)), mode='constant')


def normalize_clip(clip):
    clip = clip - np.mean(clip)
    peak = float(np.max(np.abs(clip))) if len(clip) else 0.0
    if peak > 0.999:
        clip = clip / peak * 0.999
    return clip.astype(np.float32)


def write_clip(path, clip):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), clip, SAMPLE_RATE, subtype='PCM_16')


def find_onsets(y):
    """Find onset candidates in audio using multi-method detection."""
    if y is None or len(y) < 100:
        return []
    abs_y = np.abs(y)
    
    # Onset detection
    hop = 128
    onset_env = librosa.onset.onset_strength(y=y, sr=SAMPLE_RATE, hop_length=hop)
    onset_frames = librosa.onset.onset_detect(
        onset_envelope=onset_env, sr=SAMPLE_RATE, hop_length=hop,
        backtrack=False, delta=0.1, wait=2
    )
    candidates = [int(f * hop) for f in onset_frames]
    candidates.append(int(np.argmax(abs_y)))  # Peak
    
    # Score and filter
    diff = np.abs(np.diff(y, prepend=0.0))
    scores = {c: float(abs_y[c]) + float(diff[c]) for c in candidates if 0 <= c < len(y)}
    sorted_c = [c for c, _ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)]
    
    min_gap = max(TARGET_SAMPLES // 2, int(SAMPLE_RATE * 0.03))
    filtered = []
    for c in sorted_c:
        if all(abs(c - kept) >= min_gap for kept in filtered):
            filtered.append(c)
        if len(filtered) >= 8:
            break
    return filtered


print('✅ Helper functions defined.')

In [ ]:
# ============================================================
# CELL 5: Heuristic Classifier (no ML model needed)
# ============================================================

def classify_clip(clip):
    """
    Classify a 250ms clip as 'gunshot', 'nongunshot', or 'uncertain'
    using audio features (heuristic-based, no ML model).
    
    Returns: (label: str, confidence: float, metrics: dict)
    """
    abs_clip = np.abs(clip)
    peak = float(np.max(abs_clip)) if len(abs_clip) else 0.0
    rms = float(np.sqrt(np.mean(np.square(clip)))) if len(clip) else 0.0
    crest = peak / (rms + 1e-12)
    median_abs = float(np.median(abs_clip)) if len(abs_clip) else 0.0
    prominence = peak / (median_abs + 1e-12)
    
    # Attack ratio
    attack = float(np.max(np.abs(np.diff(clip)))) if len(clip) > 1 else 0.0
    attack_ratio = attack / (rms + 1e-12)
    
    # Spectral centroid
    try:
        centroid = float(np.mean(librosa.feature.spectral_centroid(y=clip, sr=SAMPLE_RATE)[0]))
    except Exception:
        centroid = 0.0
    
    # ZCR
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y=clip)[0]))
    
    metrics = {
        'peak': peak, 'rms': rms, 'crest_factor': crest,
        'prominence': prominence, 'attack_ratio': attack_ratio,
        'spectral_centroid': centroid, 'zcr': zcr,
    }
    
    # --- Silence ---
    if peak < 0.003 or rms < 0.0005:
        return 'nongunshot', 0.9, metrics
    
    # --- Gunshot heuristic ---
    # A gunshot has: high crest, high peak, broadband energy, sharp attack
    gunshot_score = 0.0
    if crest >= GUNSHOT_CREST_MIN:
        gunshot_score += 0.25
    if peak >= GUNSHOT_PEAK_MIN:
        gunshot_score += 0.2
    if centroid >= GUNSHOT_CENTROID_MIN:
        gunshot_score += 0.2
    if prominence >= 4.0:
        gunshot_score += 0.15
    if attack_ratio >= 5.0:
        gunshot_score += 0.2
    
    if gunshot_score >= 0.7:
        return 'gunshot', gunshot_score, metrics
    elif gunshot_score >= 0.4:
        return 'uncertain', gunshot_score, metrics
    else:
        return 'nongunshot', 1.0 - gunshot_score, metrics


print('✅ Heuristic classifier defined.')

In [ ]:
# ============================================================
# CELL 6: BATCH MODE
# ============================================================

if MODE == 'batch':
    print(f'\n{"=" * 65}')
    print(f'BATCH MODE — Processing folder: {BATCH_INPUT_DIR}')
    print(f'{"=" * 65}\n')
    
    assert BATCH_INPUT_DIR.exists(), f'❌ Input dir not found: {BATCH_INPUT_DIR}'
    
    # Setup output
    batch_out = OUTPUT_DIR / 'batch_output'
    if batch_out.exists() and OVERWRITE:
        shutil.rmtree(batch_out)
    
    gs_dir = batch_out / 'gunshot'
    ngs_dir = batch_out / 'nongunshot'
    unc_dir = batch_out / 'uncertain'
    reports_dir = OUTPUT_DIR / 'reports'
    
    for d in [gs_dir, ngs_dir, unc_dir, reports_dir]:
        d.mkdir(parents=True, exist_ok=True)
    
    source_files = collect_wavs([BATCH_INPUT_DIR])
    print(f'Found {len(source_files):,} .wav files\n')
    
    manifest_rows = []
    counts = {'gunshot': 0, 'nongunshot': 0, 'uncertain': 0}
    clip_idx = 0
    hop_samples = int(SAMPLE_RATE * BATCH_HOP_MS / 1000)
    
    for src_path in tqdm(source_files, desc='📦 Batch processing', unit='file'):
        try:
            y = load_audio(src_path)
        except Exception:
            continue
        
        if len(y) < 4:
            continue
        
        src_key = sanitize_name(src_path.stem)
        
        # Try onset detection first
        onsets = find_onsets(y)
        
        # Extract clips at onsets
        clips_extracted = []
        for k, center in enumerate(onsets):
            start = int(center) - PRE_EVENT_SAMPLES
            end = start + TARGET_SAMPLES
            pad_left = max(0, -start)
            pad_right = max(0, end - len(y))
            clip = y[max(0, start):min(len(y), end)]
            if pad_left > 0 or pad_right > 0:
                clip = np.pad(clip, (pad_left, pad_right))
            clip = force_exact_length(clip)
            clip = normalize_clip(clip)
            clips_extracted.append((clip, f'onset_k{k:02d}'))
        
        # Also do sliding window for coverage
        if len(y) >= TARGET_SAMPLES:
            starts = list(range(0, len(y) - TARGET_SAMPLES + 1, hop_samples))
            random.shuffle(starts)
            for w_idx, s in enumerate(starts[:20]):  # Cap at 20 windows
                clip = y[s:s + TARGET_SAMPLES]
                clip = force_exact_length(clip)
                clip = normalize_clip(clip)
                # Only add if not overlapping with onset clips
                is_duplicate = False
                for onset_c in onsets:
                    if abs(s - (onset_c - PRE_EVENT_SAMPLES)) < TARGET_SAMPLES // 2:
                        is_duplicate = True
                        break
                if not is_duplicate:
                    clips_extracted.append((clip, f'window_w{w_idx:03d}'))
        
        # Classify and save each clip
        for clip, tag in clips_extracted:
            label, conf, metrics = classify_clip(clip)
            clip_idx += 1
            
            if label == 'gunshot':
                out_path = gs_dir / f'gs_{clip_idx:07d}_{src_key}_{tag}.wav'
            elif label == 'uncertain':
                out_path = unc_dir / f'unc_{clip_idx:07d}_{src_key}_{tag}.wav'
            else:
                out_path = ngs_dir / f'ngs_{clip_idx:07d}_{src_key}_{tag}.wav'
            
            write_clip(out_path, clip)
            counts[label] += 1
            
            manifest_rows.append({
                'output_path': str(out_path.relative_to(OUTPUT_DIR)),
                'source_path': str(src_path),
                'label': label,
                'confidence': conf,
                'tag': tag,
                **metrics,
            })
    
    # Save reports
    manifest_df = pd.DataFrame(manifest_rows)
    manifest_df.to_csv(reports_dir / 'manifest.csv', index=False)
    summary = {'mode': 'batch', 'input': str(BATCH_INPUT_DIR), 'counts': counts, 'total': clip_idx}
    (reports_dir / 'summary.json').write_text(json.dumps(summary, indent=2))
    
    print(f'\n{"=" * 50}')
    print(f'BATCH COMPLETE')
    print(f'{"=" * 50}')
    print(f'🔫 Gunshot     : {counts["gunshot"]:,}')
    print(f'🎵 Non-gunshot : {counts["nongunshot"]:,}')
    print(f'❓ Uncertain   : {counts["uncertain"]:,}')
    print(f'📊 Total       : {clip_idx:,}')
    print(f'{"=" * 50}')

else:
    print(f'⏭️ Skipping batch mode (MODE = "{MODE}")')

In [ ]:
# ============================================================
# CELL 7: SINGLE FILE MODE
# ============================================================

if MODE == 'single':
    assert SINGLE_INPUT_FILE is not None, '❌ Set SINGLE_INPUT_FILE in Cell 3!'
    assert Path(SINGLE_INPUT_FILE).exists(), f'❌ File not found: {SINGLE_INPUT_FILE}'
    
    print(f'\n{"=" * 65}')
    print(f'SINGLE FILE MODE — Analyzing: {Path(SINGLE_INPUT_FILE).name}')
    print(f'{"=" * 65}\n')
    
    single_out = OUTPUT_DIR / 'single_output'
    if single_out.exists() and OVERWRITE:
        shutil.rmtree(single_out)
    clips_dir = single_out / 'clips'
    clips_dir.mkdir(parents=True, exist_ok=True)
    
    y = load_audio(SINGLE_INPUT_FILE)
    duration_s = len(y) / SAMPLE_RATE
    
    print(f'Duration  : {duration_s:.2f}s ({len(y):,} samples)')
    print(f'Peak      : {np.max(np.abs(y)):.4f}')
    print(f'RMS       : {np.sqrt(np.mean(y**2)):.4f}')
    
    # Find onsets
    onsets = find_onsets(y)
    print(f'\nOnsets detected: {len(onsets)}')
    
    # Plot full waveform with onset markers
    fig, axes = plt.subplots(3, 1, figsize=(16, 10))
    
    # Waveform
    librosa.display.waveshow(y, sr=SAMPLE_RATE, ax=axes[0])
    axes[0].set_title(f'Waveform: {Path(SINGLE_INPUT_FILE).name}')
    for i, onset in enumerate(onsets):
        t = onset / SAMPLE_RATE
        axes[0].axvline(x=t, color='r', linestyle='--', alpha=0.7)
        axes[0].text(t, axes[0].get_ylim()[1] * 0.9, f'#{i+1}', color='r', fontsize=8)
    
    # Mel spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=SAMPLE_RATE, n_mels=128)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, sr=SAMPLE_RATE, x_axis='time', y_axis='mel', ax=axes[1])
    axes[1].set_title('Mel-Spectrogram')
    
    # Onset strength envelope
    onset_env = librosa.onset.onset_strength(y=y, sr=SAMPLE_RATE)
    times = librosa.times_like(onset_env, sr=SAMPLE_RATE)
    axes[2].plot(times, onset_env, label='Onset Strength')
    axes[2].set_title('Onset Strength Envelope')
    axes[2].set_xlabel('Time (s)')
    axes[2].legend()
    
    plt.tight_layout()
    plt.savefig(single_out / 'analysis.png', dpi=150)
    plt.show()
    
    # Extract and classify each onset clip
    print(f'\n{"=" * 50}')
    print(f'EXTRACTED CLIPS')
    print(f'{"=" * 50}')
    
    for k, center in enumerate(onsets):
        start = int(center) - PRE_EVENT_SAMPLES
        end = start + TARGET_SAMPLES
        pad_left = max(0, -start)
        pad_right = max(0, end - len(y))
        clip = y[max(0, start):min(len(y), end)]
        if pad_left > 0 or pad_right > 0:
            clip = np.pad(clip, (pad_left, pad_right))
        clip = force_exact_length(clip)
        clip = normalize_clip(clip)
        
        label, conf, metrics = classify_clip(clip)
        t_s = center / SAMPLE_RATE
        
        out_name = f'clip_{k+1:03d}_{label}_{conf:.2f}.wav'
        write_clip(clips_dir / out_name, clip)
        
        icon = '🔫' if label == 'gunshot' else '🎵' if label == 'nongunshot' else '❓'
        print(f'  {icon} Clip #{k+1} @ {t_s:.3f}s — {label} (conf: {conf:.2f}) '
              f'[peak={metrics["peak"]:.3f}, crest={metrics["crest_factor"]:.1f}, '
              f'centroid={metrics["spectral_centroid"]:.0f}Hz]')
    
    # Play original audio
    print(f'\n🔊 Full audio:')
    display(ipd.Audio(str(SINGLE_INPUT_FILE)))

else:
    print(f'⏭️ Skipping single file mode (MODE = "{MODE}")')

In [ ]:
# ============================================================
# CELL 8: MICROPHONE MODE
# ============================================================

if MODE == 'microphone':
    assert SD_AVAILABLE, '❌ sounddevice not installed! Run: pip install sounddevice'
    
    print(f'\n{"=" * 65}')
    print(f'MICROPHONE MODE — Live Capture')
    print(f'{"=" * 65}')
    
    # List available devices
    print(f'\nAvailable audio devices:')
    print(sd.query_devices())
    print(f'\nUsing device: {MIC_DEVICE_INDEX or "default"}')
    
    mic_out = OUTPUT_DIR / 'mic_captures'
    if mic_out.exists() and OVERWRITE:
        shutil.rmtree(mic_out)
    mic_out.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / 'reports').mkdir(parents=True, exist_ok=True)
    
    # Rolling buffer
    buffer_samples = int(SAMPLE_RATE * MIC_BUFFER_SECONDS)
    audio_buffer = np.zeros(buffer_samples, dtype=np.float32)
    event_count = 0
    last_capture_time = 0.0
    is_running = True
    events_log = []
    
    def audio_callback(indata, frames, time_info, status):
        """Called by sounddevice for each audio chunk."""
        nonlocal audio_buffer, event_count, last_capture_time
        
        chunk = indata[:, 0].astype(np.float32)  # Mono
        
        # Shift buffer and append new data
        audio_buffer = np.roll(audio_buffer, -len(chunk))
        audio_buffer[-len(chunk):] = chunk
        
        # Check trigger
        rms = float(np.sqrt(np.mean(chunk ** 2)))
        current_time = time.time()
        
        if rms > MIC_TRIGGER_RMS and (current_time - last_capture_time) > MIC_COOLDOWN_S:
            last_capture_time = current_time
            event_count += 1
            
            # Extract 250ms clip centered on the loudest sample in recent buffer
            recent = audio_buffer[-int(SAMPLE_RATE * 0.5):]  # Last 500ms
            peak_idx = int(np.argmax(np.abs(recent)))
            center = len(audio_buffer) - int(SAMPLE_RATE * 0.5) + peak_idx
            
            start = center - PRE_EVENT_SAMPLES
            end = start + TARGET_SAMPLES
            start = max(0, start)
            end = min(len(audio_buffer), end)
            
            clip = audio_buffer[start:end].copy()
            clip = force_exact_length(clip)
            clip = normalize_clip(clip)
            
            # Classify
            label, conf, metrics = classify_clip(clip)
            
            # Save
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            out_name = f'event_{event_count:03d}_{timestamp}_{label}.wav'
            out_path = mic_out / out_name
            write_clip(out_path, clip)
            
            events_log.append({
                'event': event_count,
                'timestamp': timestamp,
                'rms_trigger': rms,
                'label': label,
                'confidence': conf,
                'file': out_name,
                **metrics,
            })
            
            icon = '🔫' if label == 'gunshot' else '🎵' if label == 'nongunshot' else '❓'
            print(f'  {icon} Event #{event_count} [{timestamp}] — {label} '
                  f'(conf: {conf:.2f}, rms: {rms:.4f})')
    
    # Start capture
    print(f'\n🎤 Listening... (Ctrl+C or wait {MIC_CAPTURE_DURATION_S}s to stop)\n')
    
    try:
        with sd.InputStream(
            device=MIC_DEVICE_INDEX,
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype='float32',
            blocksize=int(SAMPLE_RATE * 0.1),  # 100ms blocks
            callback=audio_callback
        ):
            if MIC_CAPTURE_DURATION_S > 0:
                time.sleep(MIC_CAPTURE_DURATION_S)
            else:
                print('   Recording indefinitely. Press Ctrl+C to stop.')
                while True:
                    time.sleep(0.5)
    except KeyboardInterrupt:
        print('\n\n⏹️ Stopped by user.')
    
    # Save events log
    if events_log:
        events_df = pd.DataFrame(events_log)
        events_df.to_csv(OUTPUT_DIR / 'reports' / 'mic_events.csv', index=False)
    
    print(f'\n{"=" * 50}')
    print(f'CAPTURE COMPLETE')
    print(f'{"=" * 50}')
    print(f'Total events captured: {event_count}')
    print(f'Saved to: {mic_out}')
    print(f'{"=" * 50}')

else:
    print(f'⏭️ Skipping microphone mode (MODE = "{MODE}")')

In [ ]:
# ============================================================
# CELL 9: TEST MICROPHONE (run this to verify your mic works)
# ============================================================

if SD_AVAILABLE:
    print('\n🎤 Microphone Test — Recording 2 seconds...\n')
    
    test_duration = 2.0  # seconds
    try:
        test_audio = sd.rec(
            int(test_duration * SAMPLE_RATE),
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype='float32',
            device=MIC_DEVICE_INDEX
        )
        sd.wait()
        test_audio = test_audio.flatten()
        
        peak = np.max(np.abs(test_audio))
        rms = np.sqrt(np.mean(test_audio ** 2))
        
        print(f'  Peak amplitude : {peak:.4f}')
        print(f'  RMS energy     : {rms:.4f}')
        
        if peak < 0.001:
            print(f'  ⚠️ Very low signal! Check your microphone settings.')
        else:
            print(f'  ✅ Microphone is working!')
        
        # Plot
        fig, ax = plt.subplots(1, 1, figsize=(12, 3))
        librosa.display.waveshow(test_audio, sr=SAMPLE_RATE, ax=ax)
        ax.set_title(f'Mic Test ({test_duration}s)')
        plt.tight_layout()
        plt.show()
        
        print('\n🔊 Playback:')
        display(ipd.Audio(test_audio, rate=SAMPLE_RATE))
        
    except Exception as e:
        print(f'  ❌ Error: {e}')
        print(f'  Try setting MIC_DEVICE_INDEX to a specific device number.')
        print(f'  Available devices:')
        print(sd.query_devices())
else:
    print('⚠️ sounddevice not available. Install with: pip install sounddevice')

In [ ]:
# ============================================================
# CELL 10: RESULTS SUMMARY & VISUAL AUDIT
# ============================================================

print(f'\n{"=" * 65}')
print(f'RESULTS SUMMARY')
print(f'{"=" * 65}\n')

# Check what output exists
for subdir in ['batch_output', 'single_output', 'mic_captures']:
    d = OUTPUT_DIR / subdir
    if d.exists():
        all_wavs = list(d.rglob('*.wav'))
        print(f'📂 {subdir}/ : {len(all_wavs):,} clips')
        
        # Show subfolders
        for sub in sorted(d.iterdir()):
            if sub.is_dir():
                sub_count = len(list(sub.glob('*.wav')))
                print(f'    ├─ {sub.name}/ : {sub_count:,}')

# Plot random samples if batch output exists
batch_gs = OUTPUT_DIR / 'batch_output' / 'gunshot'
batch_ngs = OUTPUT_DIR / 'batch_output' / 'nongunshot'

if batch_gs.exists() or batch_ngs.exists():
    gs_files = list(batch_gs.glob('*.wav')) if batch_gs.exists() else []
    ngs_files = list(batch_ngs.glob('*.wav')) if batch_ngs.exists() else []
    
    n_gs = min(3, len(gs_files))
    n_ngs = min(3, len(ngs_files))
    
    if n_gs + n_ngs > 0:
        fig, axes = plt.subplots(n_gs + n_ngs, 2, figsize=(14, 3 * (n_gs + n_ngs)))
        if n_gs + n_ngs == 1:
            axes = axes.reshape(1, -1)
        
        row = 0
        for f in random.sample(gs_files, n_gs):
            y, sr = librosa.load(f, sr=None)
            librosa.display.waveshow(y, sr=sr, ax=axes[row, 0])
            axes[row, 0].set_title(f'🔫 {f.name[:40]}', fontsize=9)
            S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
            librosa.display.specshow(librosa.power_to_db(S, ref=np.max),
                                    sr=sr, x_axis='time', y_axis='mel', ax=axes[row, 1])
            row += 1
        
        for f in random.sample(ngs_files, n_ngs):
            y, sr = librosa.load(f, sr=None)
            librosa.display.waveshow(y, sr=sr, ax=axes[row, 0])
            axes[row, 0].set_title(f'🎵 {f.name[:40]}', fontsize=9)
            S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
            librosa.display.specshow(librosa.power_to_db(S, ref=np.max),
                                    sr=sr, x_axis='time', y_axis='mel', ax=axes[row, 1])
            row += 1
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'reports' / 'results_audit.png', dpi=150)
        plt.show()

print(f'\n✅ All output saved to: {OUTPUT_DIR}')